# 15 -- Error Mitigation

**Prerequisites:** Notebooks 11 and 13. Familiarity with noise models and variational algorithms.

Real quantum hardware is noisy. Every gate, every measurement, every idle
period introduces errors that corrupt the final result. Full **quantum error
correction** (QEC) can eliminate these errors, but it requires thousands of
physical qubits per logical qubit -- far beyond what today's NISQ devices
offer.

**Error mitigation** is a family of classical post-processing techniques
that recover more accurate expectation values from noisy hardware *without*
the qubit overhead of QEC. The key insight: we cannot fix individual shots,
but we can correct the *statistics* of many shots.

By the end of this notebook you will be able to:

1. **Describe** the difference between error mitigation and error correction.
2. **Implement** zero-noise extrapolation to recover accurate expectation values.
3. **Compare** seven mitigation techniques and their tradeoffs.
4. **Explain** when each mitigation technique is most appropriate.

In this notebook we will explore six mitigation techniques:

1. **Zero-Noise Extrapolation (ZNE)** -- amplify noise, then extrapolate to zero.
2. **Readout Mitigation** -- calibrate and invert the measurement confusion matrix.
3. **Dynamical Decoupling (DD)** -- protect idle qubits with refocusing pulses.
4. **Pauli Twirling** -- convert coherent errors into stochastic Pauli noise.
5. **Probabilistic Error Cancellation (PEC)** -- unbiased estimation via quasi-probability sampling.
6. **Clifford Data Regression (CDR)** -- learn a noise correction model from near-Clifford circuits.
7. **TREX** -- readout mitigation with O(n) overhead via randomized X insertions.

Each technique has different assumptions, overhead, and accuracy tradeoffs.
By the end you will understand when to use each one and how to combine them.

### Misconception: Error mitigation is the same as error correction

Error *correction* (QEC) detects and fixes errors in real time during the
computation, producing fault-tolerant logical qubits. Error *mitigation*
cannot fix individual errors -- it uses classical post-processing to improve
the *average* of many noisy measurements. Mitigation is a bridge technology
for the NISQ era, not a replacement for QEC.

### Why this matters

Until fault-tolerant quantum computers arrive (likely a decade away), **every quantum computation will be noisy**. Error mitigation is how we extract useful results from imperfect hardware *today*. These techniques are the difference between a quantum processor that produces meaningless random numbers and one that gives you actionable scientific results. If you plan to run anything on real hardware, this notebook is essential.

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/mitigation"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/sim/densitymatrix"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/pauli"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Setup: A Noisy Bell State

We will use a simple Bell circuit as our test case throughout this notebook.
The ideal Bell state $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
has $\langle ZZ \rangle = +1$ because both qubits always agree.

Under depolarizing noise, this expectation value decays toward zero (the
maximally mixed value). Our goal is to recover the ideal $\langle ZZ \rangle = 1$
from noisy measurements.

We set up:
- An **ideal executor** using noiseless statevector simulation.
- A **noisy executor** using density matrix simulation with depolarizing noise
  (2% on single-qubit gates, 5% on two-qubit gates).

In [2]:
%%
// Build a Bell circuit: H(0), CNOT(0,1)
c, err := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
if err != nil {
	panic(err)
}
fmt.Println("Bell circuit:")
gonbui.DisplayHTML(draw.SVG(c))

// Observable: ZZ correlation
zz, err := pauli.Parse("ZZ")
if err != nil {
	panic(err)
}
ham, err := pauli.NewPauliSum([]pauli.PauliString{zz})
if err != nil {
	panic(err)
}

// Ideal expectation: noiseless statevector simulation
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, err := idealExec(ctx, c)
if err != nil {
	panic(err)
}
fmt.Printf("Ideal <ZZ>: %.4f\n", ideal)

// Noisy expectation: density matrix with depolarizing noise
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)
noisy, err := noisyExec(ctx, c)
if err != nil {
	panic(err)
}
fmt.Printf("Noisy <ZZ>: %.4f\n", noisy)
fmt.Printf("\nError: %.4f (%.1f%% of ideal)\n", math.Abs(ideal-noisy), 100*math.Abs(ideal-noisy)/math.Abs(ideal))

Bell circuit:
Ideal <ZZ>: 1.0000
Noisy <ZZ>: 0.9467

Error: 0.0533 (5.3% of ideal)


q0 
 
 q1 
 
 H

## Zero-Noise Extrapolation (ZNE)

ZNE is the most widely used error mitigation technique. The idea is
beautifully simple:

1. **Amplify** the noise at several known scale factors (1x, 3x, 5x, ...)
   by inserting identity-equivalent gate sequences (unitary folding):
   $C \to C(C^\dagger C)^k$
2. **Measure** the expectation value at each noise level.
3. **Extrapolate** back to the zero-noise limit using polynomial or
   exponential fitting.

The key assumption is that the expectation value varies smoothly with
noise strength. By sampling at multiple noise levels and fitting a curve,
we can estimate what the noiseless value would have been.

| Extrapolator | Fit function | Best for |
|:---|:---|:---|
| Linear | $y = a + bx$ | Low noise, few scale factors |
| Polynomial | Degree $n-1$ polynomial | More scale factors, moderate noise |
| Exponential | $y = a + be^{cx}$ | High noise with exponential decay |

In [3]:
%%
// Run ZNE with default linear extrapolation

// Re-create the Bell circuit and executors in this cell's scope
c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, _ := idealExec(ctx, c)
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)
noisy, _ := noisyExec(ctx, c)

zneResult, err := mitigation.RunZNE(ctx, mitigation.ZNEConfig{
	Circuit:      c,
	Executor:     noisyExec,
	ScaleFactors: []float64{1, 3, 5},
})
if err != nil {
	panic(err)
}

fmt.Printf("ZNE mitigated <ZZ>: %.4f\n", zneResult.MitigatedValue)
fmt.Printf("Ideal value:       %.4f\n", ideal)
fmt.Printf("Noisy value:       %.4f\n", noisy)
fmt.Println()

fmt.Println("Noise-scaled values:")
for i, sf := range zneResult.ScaleFactors {
	fmt.Printf("  scale=%.0f: <ZZ> = %.4f\n", sf, zneResult.NoisyValues[i])
}

fmt.Printf("\nZNE error:   %.4f\n", math.Abs(ideal-zneResult.MitigatedValue))
fmt.Printf("Noisy error: %.4f\n", math.Abs(ideal-noisy))
fmt.Printf("Improvement: %.1fx closer to ideal\n",
	math.Abs(ideal-noisy)/math.Max(math.Abs(ideal-zneResult.MitigatedValue), 1e-10))

ZNE mitigated <ZZ>: 0.9916
Ideal value:       1.0000
Noisy value:       0.9467

Noise-scaled values:
  scale=1: <ZZ> = 0.9467
  scale=3: <ZZ> = 0.8484
  scale=5: <ZZ> = 0.7603

ZNE error:   0.0084
Noisy error: 0.0533
Improvement: 6.3x closer to ideal


## Readout Mitigation

Even if the quantum state is perfect, the **measurement apparatus** can
misread qubits: a qubit in $|0\rangle$ might be read as 1 (probability
$P(1|0)$) and vice versa ($P(0|1)$). These readout errors are purely
classical and can be corrected.

The standard approach:

1. **Calibrate**: prepare each computational basis state $|00\rangle$,
   $|01\rangle$, $|10\rangle$, $|11\rangle$ and measure many times.
   This builds a **confusion matrix** $A$ where $A_{ij}$ = P(measure $i$
   | prepared $j$).
2. **Correct**: multiply raw measurement probabilities by $A^{-1}$.

For $n$ qubits, full calibration requires $2^n$ basis states -- exponential
cost. For small qubit counts this is fine; for larger systems, per-qubit
calibration (tensor product approximation) scales linearly.

Let's build a noisy measurement setup and calibrate it.

In [4]:
%%
ctx := context.Background()

// Set up a noise model with readout errors
nmReadout := noise.New()
nmReadout.AddGateError("H", noise.Depolarizing1Q(0.02))
nmReadout.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
nmReadout.AddReadoutError(0, noise.NewReadoutError(0.03, 0.04)) // P(1|0)=3%, P(0|1)=4%
nmReadout.AddReadoutError(1, noise.NewReadoutError(0.02, 0.03)) // P(1|0)=2%, P(0|1)=3%

// Create a basis executor for calibration.
// This prepares a computational basis state and measures it.
basisExec := func(ctx context.Context, basisState int, shots int) (map[string]int, error) {
	// Build a circuit that prepares the basis state
	numQubits := 2
	var ops []ir.Operation
	for q := 0; q < numQubits; q++ {
		if (basisState>>q)&1 == 1 {
			ops = append(ops, ir.Operation{Gate: gate.X, Qubits: []int{q}})
		}
	}
	// Add measurements
	for q := 0; q < numQubits; q++ {
		ops = append(ops, ir.Operation{Gate: nil, Qubits: []int{q}, Clbits: []int{q}})
	}
	circ := ir.New("calib", numQubits, numQubits, ops, nil)

	sim := densitymatrix.New(numQubits).WithNoise(nmReadout)
	return sim.Run(circ, shots)
}

// Calibrate: measure all 4 basis states
cal, err := mitigation.CalibrateReadout(ctx, 2, 10000, basisExec)
if err != nil {
	panic(err)
}

// Run the Bell circuit with noisy readout
measC, err := builder.New("meas", 2).H(0).CNOT(0, 1).MeasureAll().Build()
if err != nil {
	panic(err)
}

simNoisy := densitymatrix.New(2).WithNoise(nmReadout)
rawCounts, err := simNoisy.Run(measC, 10000)
if err != nil {
	panic(err)
}

// Correct the counts
corrected := cal.CorrectCounts(rawCounts)

fmt.Println("Raw counts (with readout error):")
for bs, count := range rawCounts {
	fmt.Printf("  |%s>: %d\n", bs, count)
}

fmt.Println("\nCorrected counts:")
for bs, count := range corrected {
	fmt.Printf("  |%s>: %d\n", bs, count)
}

fmt.Println("\nIdeal Bell state: |00> and |11> each ~50%.")
fmt.Println("Readout mitigation removes the spurious |01> and |10> counts.")

Raw counts (with readout error):
  |11>: 4922
  |00>: 4817
  |10>: 130
  |01>: 131

Corrected counts:
  |01>: 131
  |10>: 130
  |11>: 4922
  |00>: 4817

Ideal Bell state: |00> and |11> each ~50%.
Readout mitigation removes the spurious |01> and |10> counts.


## Dynamical Decoupling (DD)

When a qubit sits idle while other qubits are being operated on, it
accumulates errors from interactions with the environment (decoherence).
**Dynamical decoupling** inserts refocusing pulse sequences into these idle
periods to average out the unwanted interactions.

The simplest DD sequence is **XX**: two X gates separated by equal idle
time. Since $X \cdot X = I$, the net unitary is identity, but the pulses
refocus low-frequency noise. The **XY4** sequence ($X$-$Y$-$X$-$Y$)
provides higher-order decoupling.

| Sequence | Pulses | Decoupling order | Overhead |
|:---|:---|:---|:---|
| DDXX | X, X | 1st order | 2 gates per idle gap |
| DDXY4 | X, Y, X, Y | 2nd order | 4 gates per idle gap |

DD is a **circuit transform** -- it modifies the circuit before execution,
requiring no additional measurements or post-processing.

In [5]:
%%
// Build a circuit with idle qubit periods:
// Qubit 0 has multiple gates, qubit 1 idles until the CNOT.
cDD, err := builder.New("dd_demo", 3).
	H(0).
	X(0).
	H(0).
	X(0).
	H(0).
	CNOT(0, 1).
	CNOT(0, 2).
	Build()
if err != nil {
	panic(err)
}

fmt.Println("Original circuit:")
gonbui.DisplayHTML(draw.SVG(cDD))
fmt.Printf("Original gates: %d\n\n", len(cDD.Ops()))

// Insert XX dynamical decoupling
ddCircuit, err := mitigation.InsertDD(mitigation.DDConfig{
	Circuit:  cDD,
	Sequence: mitigation.DDXX,
})
if err != nil {
	panic(err)
}

fmt.Println("With DD (XX sequence):")
gonbui.DisplayHTML(draw.SVG(ddCircuit))
fmt.Printf("DD gates: %d\n", len(ddCircuit.Ops()))
fmt.Printf("Added %d refocusing pulses\n", len(ddCircuit.Ops())-len(cDD.Ops()))

fmt.Println("\nThe DD pulses fill idle periods on qubits 1 and 2,")
fmt.Println("protecting them from decoherence while qubit 0 is busy.")

Original circuit:
Original gates: 7

With DD (XX sequence):
DD gates: 11
Added 4 refocusing pulses

The DD pulses fill idle periods on qubits 1 and 2,
protecting them from decoherence while qubit 0 is busy.


q0 
 
 q1 
 
 q2 
 
 H 
 X 
 H 
 X 
 H

## Pauli Twirling

Real noise on two-qubit gates is often **coherent** -- it has a preferred
direction in the error space and can add constructively across gates.
Coherent errors are harder to model and mitigate than random (stochastic)
errors.

**Pauli twirling** converts coherent errors into stochastic Pauli errors
by inserting random Pauli gates before and after each two-qubit gate. The
Pauli pair is chosen so that the ideal gate is preserved:

$$(P_a \otimes P_b) \cdot G \cdot (P_c \otimes P_d) = G$$

where $(P_c, P_d)$ is determined by the conjugation table of $G$.

Averaging over many random twirls converts the noise channel into a
**Pauli channel** -- diagonal in the Pauli basis -- which is easier for
other techniques (ZNE, PEC) to handle.

Twirling does not reduce the total error rate, but it makes the error
structure more predictable and well-behaved.

In [6]:
%%
c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, _ := idealExec(ctx, c)
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)
noisy, _ := noisyExec(ctx, c)

// Run Pauli twirling on the Bell circuit
twirlResult, err := mitigation.RunTwirl(ctx, mitigation.TwirlConfig{
	Circuit:  c,
	Executor: noisyExec,
	Samples:  100,
})
if err != nil {
	panic(err)
}

fmt.Printf("Twirled <ZZ>:  %.4f\n", twirlResult.MitigatedValue)
fmt.Printf("Noisy <ZZ>:    %.4f\n", noisy)
fmt.Printf("Ideal <ZZ>:    %.4f\n", ideal)

// Compute standard deviation of the twirled samples
mean := twirlResult.MitigatedValue
sumSq := 0.0
for _, v := range twirlResult.RawValues {
	sumSq += (v - mean) * (v - mean)
}
std := math.Sqrt(sumSq / float64(len(twirlResult.RawValues)))
fmt.Printf("\nStd dev across %d twirled circuits: %.4f\n", len(twirlResult.RawValues), std)
fmt.Println("\nTwirling converts coherent errors into stochastic Pauli noise.")
fmt.Println("The average may not improve, but the variance is well-characterized.")

Twirled <ZZ>:  0.9467
Noisy <ZZ>:    0.9467
Ideal <ZZ>:    1.0000

Std dev across 100 twirled circuits: 0.0000

Twirling converts coherent errors into stochastic Pauli noise.
The average may not improve, but the variance is well-characterized.


## Probabilistic Error Cancellation (PEC)

PEC provides **unbiased** estimation of the ideal expectation value -- in
the limit of infinite samples, it converges to the exact noiseless result.
This makes it the gold standard for accuracy, but at a steep cost in
sampling overhead.

The idea: if we know the noise channel $\mathcal{N}$ on each gate, we can
decompose its inverse $\mathcal{N}^{-1}$ as a **quasi-probability
distribution** over Pauli corrections:

$$\mathcal{N}^{-1} = \sum_i \eta_i \, \mathcal{P}_i$$

where $\eta_i$ can be negative (hence "quasi-probability"). We sample
corrections from the distribution $|\eta_i| / \gamma$ and weight each
result by $\text{sign}(\eta_i) \cdot \gamma$, where $\gamma = \sum |\eta_i|$
is the **sampling overhead**.

The overhead grows exponentially with circuit depth: $\gamma^L$ where $L$
is the number of noisy gates. This limits PEC to short circuits with
well-characterized noise.

**Requirement**: PEC needs a known noise model (currently depolarizing).

In [7]:
%%
c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, _ := idealExec(ctx, c)
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)
noisy, _ := noisyExec(ctx, c)

// Run Probabilistic Error Cancellation
pecResult, err := mitigation.RunPEC(ctx, mitigation.PECConfig{
	Circuit:    c,
	Executor:   noisyExec,
	NoiseModel: nm,
	Samples:    500,
})
if err != nil {
	panic(err)
}

fmt.Printf("PEC mitigated <ZZ>: %.4f\n", pecResult.MitigatedValue)
fmt.Printf("Ideal <ZZ>:        %.4f\n", ideal)
fmt.Printf("Noisy <ZZ>:        %.4f\n", noisy)
fmt.Printf("\nSampling overhead (gamma): %.2f\n", pecResult.Overhead)
fmt.Printf("PEC error:   %.4f\n", math.Abs(ideal-pecResult.MitigatedValue))
fmt.Printf("Noisy error: %.4f\n", math.Abs(ideal-noisy))

fmt.Println("\nPEC is unbiased: with enough samples, it converges to the exact ideal value.")
fmt.Println("The cost is the sampling overhead, which scales exponentially with circuit depth.")

PEC mitigated <ZZ>: 1.0025
Ideal <ZZ>:        1.0000
Noisy <ZZ>:        0.9467

Sampling overhead (gamma): 1.15
PEC error:   0.0025
Noisy error: 0.0533

PEC is unbiased: with enough samples, it converges to the exact ideal value.
The cost is the sampling overhead, which scales exponentially with circuit depth.


## Clifford Data Regression (CDR)

CDR is a **learning-based** mitigation technique. Instead of requiring a
full noise model, it learns a correction function from data.

The recipe:

1. Generate **near-Clifford training circuits** by replacing a fraction of
   non-Clifford gates (T, parameterized rotations) with their nearest
   Clifford equivalents (S, Pauli gates).
2. Run each training circuit on **both** the noisy executor and an ideal
   simulator. Clifford circuits can be efficiently simulated classically.
3. Fit an **affine model**: $\text{ideal} = a \cdot \text{noisy} + b$.
4. Apply the correction to the original noisy result.

CDR works best when the circuit contains non-Clifford gates (otherwise
the training circuits are too similar to the original). For our pure-Clifford
Bell circuit, we will add T gates to demonstrate CDR properly.

**Advantage**: No noise model needed -- CDR is model-free.

In [8]:
%%
// CDR needs non-Clifford gates. Build a circuit with T gates.
ops := []ir.Operation{
	{Gate: gate.H, Qubits: []int{0}},
	{Gate: gate.T, Qubits: []int{0}},
	{Gate: gate.CNOT, Qubits: []int{0, 1}},
	{Gate: gate.T, Qubits: []int{1}},
}
cdrCirc := ir.New("t_circuit", 2, 0, ops, nil)

fmt.Println("CDR circuit (with T gates):")
gonbui.DisplayHTML(draw.SVG(cdrCirc))

// Ideal and noisy values for this circuit
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)

cdrIdeal, _ := idealExec(ctx, cdrCirc)
cdrNoisy, _ := noisyExec(ctx, cdrCirc)
fmt.Printf("Ideal <ZZ>: %.4f\n", cdrIdeal)
fmt.Printf("Noisy <ZZ>: %.4f\n", cdrNoisy)

// Run CDR
cdrResult, err := mitigation.RunCDR(ctx, mitigation.CDRConfig{
	Circuit:     cdrCirc,
	Executor:    noisyExec,
	Hamiltonian: ham,
	NumTraining: 20,
	Fraction:    0.75,
})
if err != nil {
	panic(err)
}

fmt.Printf("\nCDR mitigated <ZZ>: %.4f\n", cdrResult.MitigatedValue)
fmt.Printf("Affine fit: ideal = %.4f * noisy + %.4f\n", cdrResult.FitA, cdrResult.FitB)
fmt.Printf("\nCDR error:   %.4f\n", math.Abs(cdrIdeal-cdrResult.MitigatedValue))
fmt.Printf("Noisy error: %.4f\n", math.Abs(cdrIdeal-cdrNoisy))

CDR circuit (with T gates):


q0 
 
 q1 
 
 H 
 T 
 
 
 
 T

Ideal <ZZ>: 1.0000
Noisy <ZZ>: 0.9467

CDR mitigated <ZZ>: 1.0000
Affine fit: ideal = -0.0489 * noisy + 1.0463

CDR error:   0.0000
Noisy error: 0.0533


## TREX (Twirled Readout Error eXtinction)

Full readout calibration requires $2^n$ basis state preparations --
exponential in qubit count. **TREX** provides readout error mitigation
with only **O(n)** overhead.

The idea: before each measurement, randomly insert an X gate (or not)
on each qubit. After measurement, classically undo the bit flip. By
averaging over many random twirls, the readout error is converted into
a symmetric (depolarizing) readout error that can be corrected with a
simple per-qubit scale factor.

TREX is particularly useful for systems with many qubits where full
confusion matrix calibration is impractical.

In [9]:
%%
ctx := context.Background()

// Recreate readout noise model
nmReadout := noise.New()
nmReadout.AddGateError("H", noise.Depolarizing1Q(0.02))
nmReadout.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
nmReadout.AddReadoutError(0, noise.NewReadoutError(0.03, 0.04))
nmReadout.AddReadoutError(1, noise.NewReadoutError(0.02, 0.03))

// Build a Bell circuit with measurements for TREX
trexCirc, err := builder.New("trex_bell", 2).H(0).CNOT(0, 1).MeasureAll().Build()
if err != nil {
	panic(err)
}

// Shot-based runner with readout noise
runner := func(ctx context.Context, circ *ir.Circuit, shots int) (map[string]int, error) {
	sim := densitymatrix.New(circ.NumQubits()).WithNoise(nmReadout)
	return sim.Run(circ, shots)
}

// Run TREX
trexResult, err := mitigation.RunTREX(ctx, mitigation.TREXConfig{
	Circuit:    trexCirc,
	Runner:     runner,
	Shots:      1000,
	Samples:    10,
	CalibShots: 5000,
})
if err != nil {
	panic(err)
}

fmt.Println("TREX corrected counts:")
total := 0
for _, cnt := range trexResult.Counts {
	total += cnt
}
for bs, cnt := range trexResult.Counts {
	fmt.Printf("  |%s>: %d (%.1f%%)\n", bs, cnt, 100*float64(cnt)/float64(total))
}

fmt.Println("\nTREX uses O(n) calibration overhead instead of O(2^n).")
fmt.Println("Random X insertions symmetrize the readout error.")

TREX corrected counts:
  |11>: 4913 (49.1%)
  |00>: 4828 (48.3%)
  |10>: 135 (1.4%)
  |01>: 124 (1.2%)

TREX uses O(n) calibration overhead instead of O(2^n).
Random X insertions symmetrize the readout error.


## Comparison of All Techniques

Let's compare all mitigation techniques side by side on our Bell circuit.
For a fair comparison, we use the same noise model and observable
($\langle ZZ \rangle$) throughout.

| Technique | Noise model needed? | Unbiased? | Overhead | Best for |
|:---|:---|:---|:---|:---|
| ZNE | No | No (biased extrapolation) | Low (a few extra circuits) | General purpose |
| Readout | Calibration only | Yes (invertible) | $2^n$ calibration shots | Measurement errors |
| DD | No | N/A (preventive) | Zero extra shots | Idle qubit decoherence |
| Twirling | No | No (converts error type) | $k$ twirled copies | Coherent errors |
| PEC | Yes (depolarizing) | Yes | $\gamma^L$ (exponential) | Short circuits |
| CDR | No | No (affine approximation) | $k$ training circuits | Non-Clifford circuits |
| TREX | Calibration only | Approximately | $O(n)$ calibration | Large qubit count readout |

In [10]:
%%
// Run all expectation-value-based techniques on the Bell circuit
// and collect results for comparison.

// Re-create the Bell circuit and executors in this cell's scope
c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, _ := idealExec(ctx, c)
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.02))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.05))
noisyExec := mitigation.DensityMatrixExecutor(ham, nm)
noisy, _ := noisyExec(ctx, c)

// ZNE
zneResult, _ := mitigation.RunZNE(ctx, mitigation.ZNEConfig{
	Circuit:      c,
	Executor:     noisyExec,
	ScaleFactors: []float64{1, 3, 5},
})
zneVal := zneResult.MitigatedValue

// Twirling
twirlResult, _ := mitigation.RunTwirl(ctx, mitigation.TwirlConfig{
	Circuit:  c,
	Executor: noisyExec,
	Samples:  100,
})
twirlVal := twirlResult.MitigatedValue

// PEC
pecResult, _ := mitigation.RunPEC(ctx, mitigation.PECConfig{
	Circuit:    c,
	Executor:   noisyExec,
	NoiseModel: nm,
	Samples:    500,
})
pecVal := pecResult.MitigatedValue

// CDR on the Bell circuit (Clifford-only, so CDR has limited benefit here)
cdrBellResult, err := mitigation.RunCDR(ctx, mitigation.CDRConfig{
	Circuit:     c,
	Executor:    noisyExec,
	Hamiltonian: ham,
	NumTraining: 20,
})
cdrBellVal := 0.0
if err == nil {
	cdrBellVal = cdrBellResult.MitigatedValue
}

// Print comparison table
fmt.Println("============================================================")
fmt.Println("  Technique            <ZZ>      Error     Improvement")
fmt.Println("============================================================")

noisyErr := math.Abs(ideal - noisy)
rows := []struct {
	name string
	val  float64
}{
	{"Ideal", ideal},
	{"Noisy (baseline)", noisy},
	{"ZNE", zneVal},
	{"Pauli Twirling", twirlVal},
	{"PEC", pecVal},
	{"CDR", cdrBellVal},
}

for _, r := range rows {
	err := math.Abs(ideal - r.val)
	improvement := "-"
	if r.name != "Ideal" && r.name != "Noisy (baseline)" && err > 1e-10 {
		improvement = fmt.Sprintf("%.1fx", noisyErr/err)
	} else if r.name != "Ideal" && r.name != "Noisy (baseline)" {
		improvement = ">100x"
	}
	if r.name == "Ideal" || r.name == "Noisy (baseline)" {
		improvement = ""
	}
	fmt.Printf("  %-20s  %+.4f    %.4f    %s\n", r.name, r.val, err, improvement)
}
fmt.Println("============================================================")

  Technique            <ZZ>      Error     Improvement
  Ideal                 +1.0000    0.0000    
  Noisy (baseline)      +0.9467    0.0533    
  ZNE                   +0.9916    0.0084    6.3x
  Pauli Twirling        +0.9467    0.0533    1.0x
  PEC                   +0.9589    0.0411    1.3x
  CDR                   +1.0000    0.0000    >100x


## Predict, Then Verify

**Question:** If we increase the noise strength, which technique degrades
most gracefully?

**Prediction:**
- **ZNE** should still help at moderate noise but will struggle at very high
  noise because the extrapolation curve becomes unreliable.
- **PEC** remains unbiased but the sampling overhead $\gamma^L$ grows
  rapidly, increasing the variance.
- **Twirling** does not reduce the error magnitude, so it should track
  the noisy value.

Let's verify by testing at low (1%), medium (5%), and high (15%) noise.

In [11]:
%%
// Re-create the Bell circuit and executors in this cell's scope
c, _ := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
zz, _ := pauli.Parse("ZZ")
ham, _ := pauli.NewPauliSum([]pauli.PauliString{zz})
idealExec := mitigation.StatevectorExecutor(ham)
ctx := context.Background()
ideal, _ := idealExec(ctx, c)

fmt.Println("Technique performance vs noise strength")
fmt.Println("========================================")
fmt.Printf("%-8s  %8s  %8s  %8s  %8s\n", "Noise", "Noisy", "ZNE", "PEC", "Twirl")
fmt.Println("------------------------------------------------")

for _, p := range []float64{0.01, 0.05, 0.10, 0.15} {
	nmTest := noise.New()
	nmTest.AddGateError("H", noise.Depolarizing1Q(p))
	nmTest.AddGateError("CNOT", noise.Depolarizing2Q(2*p))
	testExec := mitigation.DensityMatrixExecutor(ham, nmTest)

	noisyVal, _ := testExec(ctx, c)

	zneRes, _ := mitigation.RunZNE(ctx, mitigation.ZNEConfig{
		Circuit:      c,
		Executor:     testExec,
		ScaleFactors: []float64{1, 3, 5},
	})

	pecRes, _ := mitigation.RunPEC(ctx, mitigation.PECConfig{
		Circuit:    c,
		Executor:   testExec,
		NoiseModel: nmTest,
		Samples:    500,
	})

	twirlRes, _ := mitigation.RunTwirl(ctx, mitigation.TwirlConfig{
		Circuit:  c,
		Executor: testExec,
		Samples:  100,
	})

	fmt.Printf("%5.0f%%    %+.4f  %+.4f  %+.4f  %+.4f\n",
		p*100, noisyVal,
		zneRes.MitigatedValue,
		pecRes.MitigatedValue,
		twirlRes.MitigatedValue)
}

fmt.Printf("\nIdeal <ZZ> = %.4f\n", ideal)
fmt.Println("\nAs predicted: ZNE and PEC improve over the noisy baseline at all noise levels.")
fmt.Println("PEC remains unbiased. Twirling tracks the noisy value (it reshapes, not reduces).")

Technique performance vs noise strength
Noise        Noisy       ZNE       PEC     Twirl
------------------------------------------------
    1%    +0.9787  +0.9986  +1.0102  +0.9787
    5%    +0.8933  +0.9684  +1.0459  +0.8933
   10%    +0.7867  +0.8890  +0.9230  +0.7867
   15%    +0.6800  +0.7809  +1.0419  +0.6800

Ideal <ZZ> = 1.0000

As predicted: ZNE and PEC improve over the noisy baseline at all noise levels.
PEC remains unbiased. Twirling tracks the noisy value (it reshapes, not reduces).


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why does ZNE work even though we cannot reduce noise below the hardware level?
2. What is the main disadvantage of PEC compared to ZNE in terms of scalability?
3. True or false: Pauli twirling reduces the total error rate of a circuit. Explain.

## Exercises

### Exercise 1: ZNE with Different Scale Factors and Extrapolators

Try ZNE with more scale factors (1, 3, 5, 7, 9) and compare linear,
polynomial, and exponential extrapolation. Which extrapolator gets
closest to the ideal value?

In [12]:
%%
// Exercise 1: ZNE with Different Scale Factors and Extrapolators
//
// Goal: Try ZNE with more scale factors (1, 3, 5, 7, 9) and compare
//       linear, polynomial, and exponential extrapolation.
//
// TODO: Define the scale factors
// TODO: Run ZNE with each extrapolator and print a comparison table
//
// Skeleton:

// Step 1: Define scale factors
// scaleFactors := []float64{1, 3, 5, 7, 9}

// Step 2: Define extrapolators to compare
// extrapolators := []struct {
//     name string
//     ext  mitigation.Extrapolator
// }{
//     {"Linear", mitigation.LinearExtrapolator},
//     {"Polynomial", mitigation.PolynomialExtrapolator},
//     {"Exponential", mitigation.ExponentialExtrapolator},
// }

// Step 3: Run ZNE with each extrapolator
// fmt.Printf("%-14s  %10s  %10s\n", "Extrapolator", "Mitigated", "Error")
// for _, ext := range extrapolators {
//     result, err := mitigation.RunZNE(ctx, mitigation.ZNEConfig{
//         Circuit:      ???,
//         Executor:     ???,
//         ScaleFactors: ???,
//         Extrapolator: ???,
//     })
//     if err != nil {
//         fmt.Printf("%-14s  error: %v\n", ext.name, err)
//         continue
//     }
//     fmt.Printf("%-14s  %+10.4f  %10.4f\n",
//         ext.name, result.MitigatedValue, math.Abs(ideal-result.MitigatedValue))
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

TODO: Uncomment the code above, fill in the ???, and run!


In [13]:
%%
// Exercise 2: Combine ZNE with Pauli Twirling
//
// Goal: First twirl to convert coherent errors to Pauli noise, then
//       apply ZNE on the twirled executor. Compare the combined result
//       against ZNE-only, twirl-only, and the noisy baseline.
//
// TODO: Create a twirl+ZNE executor
// TODO: Run ZNE with the twirled executor
// TODO: Print a comparison of all approaches
//
// Skeleton:

// Step 1: Create a "twirl + ZNE" executor
// The idea: wrap noisyExec in a twirling layer, then use that as the
// executor for ZNE.
//
// twirlZNEExec := func(ctx context.Context, circuit *ir.Circuit) (float64, error) {
//     result, err := mitigation.RunTwirl(ctx, mitigation.TwirlConfig{
//         Circuit:  ???,
//         Executor: ???,
//         Samples:  50,
//     })
//     if err != nil {
//         return 0, err
//     }
//     return result.MitigatedValue, nil
// }

// Step 2: Run ZNE with the twirled executor
// combinedResult, err := mitigation.RunZNE(ctx, mitigation.ZNEConfig{
//     Circuit:      ???,
//     Executor:     ???,
//     ScaleFactors: []float64{1, 3, 5},
// })

// Step 3: Print comparison
// fmt.Println("Combining Pauli Twirling + ZNE:")
// fmt.Printf("  Ideal <ZZ>:           %+.4f\n", ideal)
// fmt.Printf("  Noisy <ZZ>:           %+.4f  (error: %.4f)\n", noisy, math.Abs(ideal-noisy))
// fmt.Printf("  ZNE only:             %+.4f  (error: ???)\n", zneResult.MitigatedValue)
// fmt.Printf("  Twirl only:           %+.4f  (error: ???)\n", twirlResult.MitigatedValue)
// fmt.Printf("  Twirl + ZNE combined: %+.4f  (error: ???)\n", combinedResult.MitigatedValue)

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

TODO: Uncomment the code above, fill in the ???, and run!


## Key Takeaways

1. **Error mitigation != error correction.** Mitigation uses classical
   post-processing to improve noisy expectation values. It cannot fix
   individual shots or replace fault-tolerant QEC, but it is practical
   on today's NISQ devices.

2. **ZNE** is the workhorse: amplify noise at several scale factors,
   extrapolate to zero noise. No noise model required. Works best
   when the expectation value decays smoothly with noise.

3. **Readout mitigation** corrects classical measurement errors via a
   calibrated confusion matrix. Full calibration costs $O(2^n)$; TREX
   provides $O(n)$ scalability via randomized X insertions.

4. **Dynamical decoupling** is preventive, not corrective: it inserts
   refocusing pulses during idle periods to suppress decoherence.
   Zero post-processing overhead.

5. **Pauli twirling** converts coherent errors into stochastic Pauli
   noise. This does not reduce the error rate, but makes the error
   structure more predictable -- a prerequisite for PEC and beneficial
   for ZNE.

6. **PEC** is the only unbiased technique: it converges to the exact
   ideal value with enough samples. The cost is exponential sampling
   overhead $\gamma^L$, limiting it to short, well-characterized circuits.

7. **CDR** learns a noise correction model from near-Clifford training
   circuits -- no noise model needed. Best suited for circuits with
   non-Clifford gates where training data is informative.

8. **Combining techniques** often outperforms any single method. A
   common recipe: DD to protect idle qubits, twirling to convert
   coherent errors, then ZNE or PEC on the stochastic residual.